# KNN modeling and the Open Asteroid Dataset
This dataset of over 800,000 known asteroids is collected from the Small Body Database maintained by the NASA's Jet Propulsion Laboratory (JPL). It contains information about each asteroid's orbit and known measurable properties. New entires are added daily and can be obtained through the NASA Open Data Portal: 
https://data.nasa.gov/dataset/jpl-small-body-database-browser 

or the JPL portal: 
https://ssd.jpl.nasa.gov/tools/sbdb_query.html

The dataset provided was obtained from Basu (2019) IJAECS, 6, 4, 2394-2835. (Feel free to use an up-to-date dataset instead!)

***The goal of this notebook is to train a KNN model to predict the classification of asteroids given some subset of available observations.*** The two classification systems we'll be testing is the classifications from the [Small Main-Belt Asteroid Spectroscopic Survey (SMASS)](http://smass.mit.edu/smass.html) and the [Tholen Asteroid Taxonomy scheme](https://ui.adsabs.harvard.edu/abs/1989aste.conf.1139T/abstract).

In [ ]:
# Importing the packages required for this exercise
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

# Data Inspection
Before you can begin building a ML model, you must first understand the data you are given and how they relate to the overarching objective of your model. Visualizing the data is always a good first step!

### Objective: 
*Start by reading in the data as a DataFrame and printing off some information, to get a feel for the type of data contained in this set. Make special note of the datatypes each entry is given as.*

In [ ]:
### INPUT NEEDED ###
# Reading in the data
asteroids = pd.read_csv( #input your file path

# Display the whole dataframe...
display(asteroids)

# ... or you can print a random sample to get a better feel for how much data is or is not available
print(asteroids.sample( # select some number of samples

asteroids.info() # Are there any values that are of an unexpected datatype? You may want to fix that!

## Header descriptions

You should notice that the dataset contains 27 unique columns. Below is a brief description of what each header means. The spectral taxonomic types that we will try to predict are kept under the headers **spec_B** and **spec_T**. 
* **name**: *Object name/designation*
* **a**: *semi-major axis [AU]*
* **e**: *eccentricity*
* **G**: *magnitude slope parameter (helps predict the magnitude of brightness as a function of the solar phase angle)*
* **i**: *inclination with respect to the xy ecliptic plane*
* **om**: *longitude of the ascending node*
* **w**: *argument of perihelion*
* **q**: *perihelion distance [AU]*
* **ad**: *aphelion distance [AU]*
* **per_y**: *Orbital period (around the sun) [yrs]*
* **data-arc**: *data arc-span [d]*
* **condition_code**: *Orbit condition code (uncertainty in orbital parameters rated from 0-9, where 0 is well-known and 9 is poorly-constrained)*
* **n_obs_used**: *number of observations used*
* **H**: *absolute magnitude parameter (brightness it would have if observed 1 AU from Sun)*
* **diameter**: *asteroid diameter [km]*
* **extent**: *object bi/tri-axial ellipsoid dimensions [km]*
* **albedo**: *geometric albedo (reflectiveness on scale of 0-1, where 1 = perfect light reflection)*
* **rot_per**: *rotation period (around rotational axis) [hrs]*
* **GM**: *standard gravitational parameter (mass times gravitational constant)*
* **BV**: *color index Bmag-Vmag*
* **UB**: *color index Umag-Bmag*
* **IR**: *color index Imag-Rmag*
* **spec_B**: *spectral taxonomic type (SMASSII)*
* **spec_T**: *spectral taxonomic type (Tholen)*
* **neo**: *Near Earth Object flag (bool)*
* **pha**: *Physically Hazardous Asteroid flag (bool)*
* **moid**: *Earth Minimum Orbit Intersection Distance [AU]*

#### Orbital Parameters

The orbits of each asteroid around the sun are described in several ways within this dataset. 

The shape of an asteroid's orbit is described by the semi-major axis (a) and eccentricity (e). The eccentricity describes how circular (e = 0) or elliptical (0 < e < 1) the orbit is. The semi-major axis is its farthest distance from the center of the circle/ellipse. The semi-minor axis (closest distance from center) can be calculated as $b = a\sqrt{1-e^{2}}$, such that larger eccentricities lead to shorter semi-minor axes (could this be useful for our models?).

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/76/An_image_describing_the_semi-major_and_semi-minor_axis_of_ellipse.svg/960px-An_image_describing_the_semi-major_and_semi-minor_axis_of_ellipse.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20141026153810" width="500"/>

(Credit: Wikimedia Commons, Sae1962)

The distance between each asteroid and the sun is described through the perihelion and aphelion distances (the minimum and maximum distance along the semi-major axis of the object's orbit), as shown below: 

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c5/Perihelion_aphelion_semimajor_axis.svg/960px-Perihelion_aphelion_semimajor_axis.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20210920195140" width="500"/>

(Credit: Wikimedia Commons, Maxmath12) 

The inclination (i), longitude of the ascending node (om), and argument of perihelion (w) together give the three-dimensional tilt in the orbit of each object, as shown below:

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/eb/Orbit1.svg/960px-Orbit1.svg.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20230508033814" width="400"/>

(Credit: Wikipedia, Lasunncty) 

#### Brightness, Color, and Classifications

Asteroids and other small bodies may be classified in several ways based on parameters such as their brightness, reflectiveness, and colors. The SMASSII and Tholen classifications are two such systems represented in this data set (**spec_B** and **spec_T**). The definition of each classification flag can be found here: https://en.wikipedia.org/wiki/Asteroid_spectral_types

The colors of objects are calculated by subtracting their absolute magnitudes measured within any 2 color filters. Represented in this dataset are  **BV** (blue vs. green), **UB** (ultra-violet vs. blue), and **IR** (infrared vs. red). Colors can be a useful means of measuring how objects interact (or emit) light, which may help determine what type of object it is. 

This example below shows a color-color diagram of small-body populations, including Kuiper belt objects (red), comets (cyan), and asteroids of different Tholen classifications (black and white circles, with classifications marked):

<img src="open_asteroid/small-body_CCD.png" width="400"/>

(Credit: Filacchione et al. 2022)

The amount of light reflected by an asteroid is described by a combination of the absolute magnitude parameter (H) and the geometric albedo (albedo). The albedo of an object is the ratio of the actual brightness we observe and that of a perfectly reflecting object of the same size, largely dependent on the surface material of the asteroid. For example, Saturn and its moon Titan receive the same amount of sunlight, but because Titan has a lower albedo, it appears dimmer: 

<img src="open_asteroid/Titan_and_Saturn_albedo.jpg" width="400"/>
(Credit: Kevin Gill) 


On the other hand, is the hypothetical brightness we would observe if the object was 1 AU from the Sun. The absolute magnitude parameter and albedo are intrinsically linked; the more reflective a given surface is, the brighter you would expect it to appear if for a given amount of sunlight.

# Feature Selection
We are building a model that can predict the SMASSII and Tholen classifications of a given asteroid using a KNN algorithm.

Based on the information provided above and within the dataset, which properties (features) would you expect to be most strongly linked to asteroid classification? Are the features the same for each classification method (SMASSII vs. Tholen)? 

### Objective: 
*Select the features that will be used for the KNN model(s) and clean up the data, removing all NaNs and infinities. Also set up your labels for each of the taxonomic classification systems.*

*NOTE: Pay close attention to the spectral classes and subclasses included in each of the classification systems. Can any of the subgroups be merged into a single class? Would that benefit your models?*

In [ ]:
### INPUT NEEDED ###
# Setting up masks to help select the features of interest
feat_list = []

### Your code here ###
# Enter any additional code needed to set up your labels and dataframes here
#

### Your code here ###
# Clean the data
#

### Objective: 
*Make a few plots to determine how many of each class are in each cleaned dataset, and the range of values they span for each selected feature. Keep in mind how this may affect your models!*

In [ ]:
### Your code here ###
#
#


# First-pass model
At this point, you should have one or two DataFrames containing a clean dataset for the SMASSII/Tholen classification model(s). It's time to build the K-Nearest Neighbors model! We'll start by feeding in the selected data into the model without making alterations to the data. Later, we can see how feature engineering may improve the results. 

### Objective:
*Train a KNN model for each classification type, based only on the features selected.*

*REMEMBER: Your data must be scaled in order for the KNN to work properly!*

In [ ]:
### INPUT NEEDED ###
# Building the training and test set
= train_test_split( , # feature DataFrame or Array, 
                    , # labels DataFrame or Array
                   test_size = , # fractional size of your test set (or use train_size)
                   random_state= ) # controls random number generator for consistent compilations

# Standardizing the feature scaling using z-scores
scaler = StandardScaler()
 = scaler.fit_transform( # Input training features and save to variable
 = scaler.fit_transform( # Input test features and save to variable

# Setting up the classifier
knn = KNeighborsClassifier( # Set up parameters for your KNN

# Training the classifier 
knn.fit( # Train the KNN on the rescaled training set

# Testing model
predict_test = knn.predict( # Test the KNN on the rescaled test set

# plotting accuracy
acc = accuracy_score( # Input expected vs. predicted labels to calculate the accuracy of the model
print(f"Accuracy: {round(acc*100, 1)}%")

### How else can you explore the accuracy of the model(s)? ###
### Do you need to repeat this for each model? ###
### Your extra code here ###
#
    

### Objective: 
*Create a series of scatterplots with different features for the x- and y-axes, with the markers colored by classification to show neighborship clustering. Can you tell by eye how why the model is clustering the asteroids as it is?*

In [ ]:
### Your code here ###
#
#

# Improving the Model

Chances are, the first-pass model isn't the most accurate or efficient model you could build for these data. There are a few techniques you can try to improve it. First, models can be optimized such that you are using the best combination of parameters when building the KNN and the most important features during the 'training' step. You can also attempt to engineer new features (as we've done before). 

## Model Optimization

You can optimize your KNN using [`sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to determine which combination of parameters from a provided list results in the best, most efficient model. It works by reading in the type of classifier and a library of parameters and potential values for those parameters. You then fit the grid to your data as you would any other `sklearn` ML model and print off the best-fit parameters. **NOTE: `GridSearchCV` may not work if you have a category that contains a small number of members compared to the number of neighbors in your KNN model.** There may be ways to work around this limitation, if you're determined to use it anyway.

### Objective: 
*Use `GridSearchCV` to optimize your KNN for each set of data (SMASSII and Tholen). Do the best parameters differ between the two? Does applying these parameters to your KNN improve your models?*

In [ ]:
### INPUT NEEDED ###
print("Creating grid...")

grid_results = GridSearchCV(KNeighborsClassifier(), 
                            {'n_neighbors': , # a list of the different numbers of neighbors to test 
                             'algorithm': ['ball_tree', 'kd_tree'], # types of algorithms to test. 'brute' is also an option
                             'leaf_size': , # a list of leaf sized to test
                             'weights': ['uniform', 'distance']} # weighting methods to test
                            )

starttime=time.perf_counter() # counter for timing
print("Fitting grid...")
grid_results.fit( # Insert your training and test data to fit
endtime = time.perf_counter()
print(f"Run time: {round((endtime-starttime)/60, 3)} mins")
print('The best model has {}'.format(grid_results.best_params_))


In [ ]:
### Test these parameters in a KNN and compare to your previous model ###
#
#

Another useful tool is [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html) from `sklearn.inspection` (which we have used before!). As a reminder, this function determines the average "importance" of each feature by applying random permutations to the features and evaluating how the model prediction changes in response. Higher mean importance values point to the most important features in the model. This can help you reduce your feature list, thereby (potentially) reducing confusion for the model.

It's worth noting that this function is much slower with KNNs than with simpler models, like linear regression, especially if you have a large dataset. The datasets used here will probably not take long to run, but for much larger sets, it's a good idea to time `permutation_importance` to decide whether it's worth running multiple times. 

### Objective:
*Determine which features are most important for determining your asteroid diameters (you may learn more about the `permutation_importance` function [here](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html), or use a different strategy to assess importance). Are there features you should remove from or add to your list? Explore how using different features change the results of your models.*

In [ ]:
### INPUT NEEDED ###

starttime = time.perf_counter() # timing permutation_importance. This may take a long time to run!
results = permutation_importance(estimator= , # your trained model
                                 X = , # data on which the importrance will be computed
                                 y = , # targets for supervised learning
                                 #scoring = , # name of scoring mechanism to use
                                 n_repeats = , # number of permutations to apply
                                 #n_jobs = , # number of parallel jobs
                                 #sample_weight = # sample weights used in scoring
                                 #max_samples = # number of samples to draw from X in each repeat
                                 random_state= ) # pseudo-random number generator

endtime = time.perf_counter()
print(f"Run time: {round((endtime-starttime)/60, 3)} mins")

# Printing an ordered list of features by their importance to the model
# NOTE: depending on what (if any) scoring mechanism chosen above, this code may not work as intended
print("\nFEATURE IMPORTANCE (mean +/- std)\n------------------\n")
for i in results.importances_mean.argsort()[::-1]:
        print(f"{feat_list[i]:<8}"
              f"{results.importances_mean[i]:.3f}"
              f" +/- {results.importances_std[i]:.3f}")


## Feature Engineering
Sometimes combining or altering the data in the set can result in a more accurate model! The results from `permutation_importance` or other sources on the SMASSII and/or Tholen system(s) may point you towards new features that can be added to your model. Keep in mind, any engineered features should be done to the original, un-scaled values and scaled afterward.

### Objective: 
*Create and test some engineered features. Compare you new model(s) to your old model(s). Which perform better for these data?* 

In [ ]:
### Your code here ###
#
#

# Applying the ML model to unclassified asteroids
Once you have a model that you're happy with, you can try it out on new data. Several of the asteroids in the dataset are missing either the SMASSII classification, the Tholen classification, or both. Ideally, a very accurate model could be used to determine their classifications automatically. 

### (OPTIONAL) Objectives: 
- *Use your model(s) to determine the spectral classes of unidentified asteroids in the original dataset.*
- *Compare the SMASS and Tholen classifications. Where do they align? Where do they differ? Can this be explained by the type and amount of data in the set?*


In [ ]:
### Your code here ###
#
#